In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ──────────────────────────────────────────────

In [5]:

# ──────────────────────────────────────────────
def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))


In [6]:
# ──────────────────────────────────────────────
# 2.  GENERATE BINARY CLASSIFICATION DATA
#     Two Gaussian blobs (class 0 and class 1)
# ──────────────────────────────────────────────
np.random.seed(42)
N = 200
X0 = np.random.randn(N, 2) + [-2, -2]   # class 0
X1 = np.random.randn(N, 2) + [ 2,  2]   # class 1
X  = np.vstack([X0, X1]).T               # shape (2, 2N)
y  = np.array([0]*N + [1]*N).reshape(1, -1)


In [7]:
# ──────────────────────────────────────────────
# 3.  NETWORK: 2 → 8 → 8 → 1
#     Hidden layers : ReLU
#     Output layer  : Sigmoid  (binary classification)
# ──────────────────────────────────────────────
def init_params():
    W1 = np.random.randn(8, 2)  * 0.1
    b1 = np.zeros((8, 1))
    W2 = np.random.randn(8, 8)  * 0.1
    b2 = np.zeros((8, 1))
    W3 = np.random.randn(1, 8)  * 0.1
    b3 = np.zeros((1, 1))
    return W1, b1, W2, b2, W3, b3

def forward(X, W1, b1, W2, b2, W3, b3):
    # ── Hidden layer 1 : ReLU ──────────────────
    Z1 = W1 @ X  + b1
    A1 = relu(Z1)                       # <-- ReLU in hidden layer

    # ── Hidden layer 2 : ReLU ──────────────────
    Z2 = W2 @ A1 + b2
    A2 = relu(Z2)                       # <-- ReLU in hidden layer

    # ── Output layer   : Sigmoid ───────────────
    Z3 = W3 @ A2 + b3
    A3 = sigmoid(Z3)                    # <-- Sigmoid in output layer
                                        #     gives P(class=1) in (0, 1)
    return Z1, A1, Z2, A2, Z3, A3

def compute_loss(A3, y):
    # Binary cross-entropy loss
    m = y.shape[1]
    loss = -np.mean(y * np.log(A3 + 1e-8) + (1 - y) * np.log(1 - A3 + 1e-8))
    return loss

def backward(X, y, Z1, A1, Z2, A2, Z3, A3, W1, W2, W3):
    m = y.shape[1]

    # ── Output layer gradient (Sigmoid + BCE) ──
    dZ3 = A3 - y                        # combined sigmoid + BCE derivative

    # ── Hidden layer 2 gradient (ReLU) ─────────
    dW3 = (dZ3 @ A2.T) / m
    db3 = np.mean(dZ3, axis=1, keepdims=True)
    dA2 = W3.T @ dZ3
    dZ2 = dA2 * relu_derivative(Z2)    # <-- ReLU derivative: 0 or 1

    # ── Hidden layer 1 gradient (ReLU) ─────────
    dW2 = (dZ2 @ A1.T) / m
    db2 = np.mean(dZ2, axis=1, keepdims=True)
    dA1 = W2.T @ dZ2
    dZ1 = dA1 * relu_derivative(Z1)    # <-- ReLU derivative: 0 or 1

    dW1 = (dZ1 @ X.T) / m
    db1 = np.mean(dZ1, axis=1, keepdims=True)

    return dW1, db1, dW2, db2, dW3, db3

def update(W1,b1,W2,b2,W3,b3, dW1,db1,dW2,db2,dW3,db3, lr=0.05):
    W1 -= lr*dW1;  b1 -= lr*db1
    W2 -= lr*dW2;  b2 -= lr*db2
    W3 -= lr*dW3;  b3 -= lr*db3
    return W1,b1,W2,b2,W3,b3


In [8]:
# ──────────────────────────────────────────────
# 4.  TRAIN
# ──────────────────────────────────────────────
W1,b1,W2,b2,W3,b3 = init_params()
losses = []

for epoch in range(1000):
    Z1,A1,Z2,A2,Z3,A3 = forward(X, W1,b1,W2,b2,W3,b3)
    loss = compute_loss(A3, y)
    losses.append(loss)
    grads = backward(X,y, Z1,A1,Z2,A2,Z3,A3, W1,W2,W3)
    W1,b1,W2,b2,W3,b3 = update(W1,b1,W2,b2,W3,b3, *grads)

# Final predictions
_,_,_,_,_,A3_final = forward(X, W1,b1,W2,b2,W3,b3)
preds    = (A3_final > 0.5).astype(int)         # threshold at 0.5
accuracy = np.mean(preds == y) * 100
print(f"Final loss     : {losses[-1]:.4f}")
print(f"Accuracy       : {accuracy:.1f}%")

Final loss     : 0.0034
Accuracy       : 100.0%
